# Residential Building Filter — wohnflaechen_nrw
**Task: Replace arbitrary P95 threshold with data-driven residential filter**

Philippe provided `wohnflaechen_nrw.parquet` — a dataset from the NRW Ministry
classifying all land parcels by use type via the `tntxt` column.

If `tntxt` contains `'Wohnbaufläche'` → confirmed residential land.
Buildings NOT on residential land can be filtered out of the clustering.

**Two tasks in this notebook:**
1. Filter buildings to confirmed residential only
2. Calculate lawn area per building using Philippe's formula
3. Complete Task 4 — Boolean geothermal feasibility flag

## Step 1 — Imports and Configuration

In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely import wkb

warnings.filterwarnings('ignore')

BASE_DIR    = os.getcwd()
FILE_M3B    = os.path.join(BASE_DIR, 'DEA_method3b_final.parquet')
FILE_WOHN   = os.path.join(BASE_DIR, 'wohnflaechen_nrw.parquet')
OUTPUT_DIR  = os.path.join(BASE_DIR, 'clustering_results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Ready')
print(f'wohnflaechen exists: {os.path.exists(FILE_WOHN)}')
print(f'M3b parquet exists : {os.path.exists(FILE_M3B)}')

## Step 2 — Load wohnflaechen Dataset

The geometry is stored as WKB bytes — decode using shapely.
The `tntxt` column format is `'Wohnbaufläche;18'` — the dataset
already contains only residential parcels (pre-filtered by Philippe).

In [ ]:
wohn = pd.read_parquet(FILE_WOHN)
print(f'Loaded: {len(wohn):,} parcels')
print(f'Columns: {wohn.columns.tolist()}')
print(f'\ntntxt sample:')
print(wohn['tntxt'].value_counts().head(5).to_string())
print(f'\nflaeche stats: min={wohn["flaeche"].min():.0f}  mean={wohn["flaeche"].mean():.0f}  max={wohn["flaeche"].max():.0f} m2')

In [ ]:
# Decode WKB geometry and set CRS
wohn['geometry_decoded'] = wohn['geometry'].apply(
    lambda x: wkb.loads(x) if isinstance(x, bytes) else x
)
wohn_gdf = gpd.GeoDataFrame(wohn, geometry='geometry_decoded', crs='EPSG:25832')
print(f'Geometry decoded: {len(wohn_gdf):,} parcels')
print(f'CRS: EPSG:25832')
print(f'Bounds: {wohn_gdf.total_bounds.round(0)}')

## Step 3 — Load Building Data

**Important CRS fix:**
The buildings parquet is labelled as EPSG:25832 but the coordinates
are in the 4-million range — this is EPSG:3035 (LAEA Europe).
We force the correct CRS and reproject to EPSG:25832 to match the wohnflaechen dataset.

In [ ]:
m3b = gpd.read_parquet(FILE_M3B, columns=['id', 'size_class', 'footprint', 'area_m2'])
m3b = m3b.set_geometry('footprint')
print(f'Buildings loaded: {len(m3b):,}')
print(f'Raw bounds: {m3b.total_bounds.round(0)}')
print(f'  → coordinates in 4M range = EPSG:3035 (mislabelled)')

# Fix: force EPSG:3035 then reproject to EPSG:25832
m3b = m3b.set_crs('EPSG:3035', allow_override=True)
m3b = m3b.to_crs('EPSG:25832')
print(f'\nReprojected to EPSG:25832')
print(f'New bounds: {m3b.total_bounds.round(0)}')
print(f'  → coordinates now in 280k-530k range ✓')

In [ ]:
# Calculate centroids for spatial join
m3b['centroid'] = m3b['footprint'].centroid
print(f'Centroids calculated: {len(m3b):,}')
print(f'Centroid bounds: {m3b["centroid"].total_bounds.round(0)}')

## Step 4 — Philippe's match_calc_area_ratio Function

Joins building centroids to parcels spatially and calculates:
```
lawn_area = (parcel_area - total_building_footprints_in_parcel) / n_buildings_in_parcel
```

In [ ]:
def match_calc_area_ratio(gdf_A, gdf_B):
    """
    Philippe's function to calculate lawn area per building.
    gdf_A: buildings with centroid + footprint geometry and area_m2
    gdf_B: parcels with flaeche (parcel area)
    Returns: matched GeoDataFrame with lawn_area column
    """
    gdf_A = gdf_A.set_geometry('centroid')
    matched_gdf = gpd.sjoin(gdf_A, gdf_B, how='left', predicate='intersects', rsuffix='lika')
    gdf_A = gdf_A.set_geometry('footprint')
    matched_gdf = matched_gdf.set_geometry('footprint')

    ogc_id_grouped_data = matched_gdf.groupby('index_lika')

    lawn_area_df = ogc_id_grouped_data.agg(
        total_osm_area=('area_m2', 'sum'),
        lika_area=('flaeche', 'first'),
        count_osm_buildings=('area_m2', 'count')
    )

    # Lawn area = (parcel area - building footprints) / number of buildings
    lawn_area_df['lawn_area'] = (
        lawn_area_df['lika_area'] - lawn_area_df['total_osm_area']
    ) / lawn_area_df['count_osm_buildings']

    matched_gdf = matched_gdf.merge(
        lawn_area_df['lawn_area'],
        left_on='index_lika',
        right_index=True,
        how='left'
    )

    return matched_gdf

print('match_calc_area_ratio() ready')

## Step 5 — Run Spatial Join and Calculate Lawn Area

This step takes 5-10 minutes on 4M buildings.

In [ ]:
print('Running spatial join...')
print('This may take 5-10 minutes on 4M buildings...')

enriched_gdf = match_calc_area_ratio(m3b, wohn_gdf)

n_matched   = enriched_gdf['lawn_area'].notna().sum()
n_unmatched = enriched_gdf['lawn_area'].isna().sum()
print(f'\nMatched to parcel    : {n_matched:,}  ({n_matched/len(enriched_gdf)*100:.1f}%)')
print(f'Not matched          : {n_unmatched:,}  ({n_unmatched/len(enriched_gdf)*100:.1f}%)')

lawn = enriched_gdf['lawn_area'].dropna()
if len(lawn) > 0:
    print(f'\nLawn area stats:')
    print(f'  min    : {lawn.min():.1f} m2')
    print(f'  mean   : {lawn.mean():.1f} m2')
    print(f'  median : {lawn.median():.1f} m2')
    print(f'  max    : {lawn.max():.1f} m2')
    print(f'  negative values: {(lawn < 0).sum():,}  (buildings larger than parcel — data issue)')
else:
    print('\nWARNING: No buildings matched — check CRS alignment')

## Step 6 — Apply Residential Filter

Buildings matched to a parcel → confirmed residential → keep
Buildings not matched → not on residential land → remove

Also clip negative lawn areas to 0 — can happen when multiple large
buildings share one small parcel.

In [ ]:
# Clip negative lawn areas to 0
enriched_gdf['lawn_area'] = enriched_gdf['lawn_area'].clip(lower=0)

# Residential flag
enriched_gdf['is_residential'] = enriched_gdf['lawn_area'].notna()

before = len(enriched_gdf)
residential_only = enriched_gdf[enriched_gdf['is_residential']].copy()
removed = before - len(residential_only)

print(f'Buildings before filter : {before:,}')
print(f'Residential buildings   : {len(residential_only):,}  ({len(residential_only)/before*100:.1f}%)')
print(f'Removed                 : {removed:,}  ({removed/before*100:.1f}%)')
print()
print('By size class:')
for sc in ['SFH', 'TH', 'MFH', 'AB']:
    b = (enriched_gdf['size_class'] == sc).sum()
    a = (residential_only['size_class'] == sc).sum()
    print(f'  {sc}: {b:>10,} → {a:>10,}  ({a/b*100:.1f}% retained)' if b > 0 else f'  {sc}: 0')

## Step 7 — Geothermal Boolean Flag

Now that we have lawn area per building we can complete Task 4.
Compare required borehole area (from lookup table) vs available lawn area.

In [ ]:
lookup_path = os.path.join(OUTPUT_DIR, 'geothermal_lookup.csv')
peak_path   = os.path.join(OUTPUT_DIR, 'peak_heat_power.parquet')

if not os.path.exists(lookup_path):
    print('geothermal_lookup.csv not found — run geothermal_lookup_table.ipynb first')
elif not os.path.exists(peak_path):
    print('peak_heat_power.parquet not found — run annual_to_peak_heat_power.ipynb first')
else:
    lookup  = pd.read_csv(lookup_path)
    peak_df = pd.read_parquet(peak_path, columns=['id', 'peak_heat_power_kw'])
    print(f'Lookup table : {len(lookup)} entries')
    print(f'Peak power   : {len(peak_df):,} buildings')

    residential_only = residential_only.merge(peak_df, on='id', how='left')

    def get_required_area(peak_kw):
        if pd.isna(peak_kw): return np.nan
        idx = (lookup['heat_power_kw'] - peak_kw).abs().idxmin()
        return lookup.loc[idx, 'area_m2']

    print('\nCalculating required borehole area...')
    residential_only['required_area_m2'] = residential_only['peak_heat_power_kw'].apply(
        get_required_area
    )

    # Boolean flag
    residential_only['geothermal_feasible'] = (
        residential_only['lawn_area'] >= residential_only['required_area_m2']
    )

    n_feasible = residential_only['geothermal_feasible'].sum()
    print(f'\nGeothermal feasibility results:')
    print(f'  Feasible (True) : {n_feasible:,}  ({n_feasible/len(residential_only)*100:.1f}%)')
    print(f'  Not feasible    : {len(residential_only)-n_feasible:,}')
    print()
    print('By size class:')
    for sc in ['SFH', 'TH', 'MFH', 'AB']:
        sub = residential_only[residential_only['size_class'] == sc]
        if sub.empty: continue
        n_f = sub['geothermal_feasible'].sum()
        print(f'  {sc}: {n_f:>8,} feasible out of {len(sub):,}  ({n_f/len(sub)*100:.1f}%)')

## Step 8 — Save Results

In [ ]:
out_cols = ['id', 'size_class', 'area_m2', 'lawn_area',
            'is_residential', 'geothermal_feasible',
            'peak_heat_power_kw', 'required_area_m2']

out_cols_available = [c for c in out_cols if c in residential_only.columns]
out_path = os.path.join(OUTPUT_DIR, 'geothermal_feasibility.parquet')
residential_only[out_cols_available].to_parquet(out_path, index=False)

print(f'Saved: {out_path}')
print(f'Rows : {len(residential_only):,}')
print()
print('Columns saved:')
for c in out_cols_available:
    print(f'  {c}')